In [ ]:
import pandas as pd
import numpy as np

idx_imagen = 3  # Choose the image: 0, 1, 2, 3 (Toucan), 4
metas_manuales = False # If False, seek the final value of PGD automatically

# Define Scenarios (sigma = 0.02 for all cases)
escenarios = [
    {'label': 'PGD CR 0.3', 'archivo': '/content/drive/MyDrive/ResultadosA2A/curves_full_pgdcr03.csv', 'metrica': 'psnr'},
    {'label': 'PGD CR 0.1', 'archivo': '/content/drive/MyDrive/ResultadosA2A/curves_full_pgdcr01.csv', 'metrica': 'psnr'},
    {'label': 'ADMM CR 0.3', 'archivo': '/content/drive/MyDrive/ResultadosA2A/curves_full_admmcr03.csv', 'metrica': 'ssim'},
    {'label': 'ADMM CR 0.1', 'archivo': '/content/drive/MyDrive/ResultadosA2A/curves_full_admmcr01.csv', 'metrica': 'ssim'}
]

print(f"Reports image index: {idx_imagen}")
print("="*60)

for esc in escenarios:
    try:
        df = pd.read_csv(esc['archivo'])
        m = esc['metrica']

        # Find the final PSNR
        pgd_img = df[(df['image'] == idx_imagen) & (df['method'] == 'PGD')].sort_values('iter')
        valor_meta = pgd_img[m].iloc[-1]

        # Find the iteration in which PGD reached the final PSNR
        t_ref = pgd_img[pgd_img[m] >= valor_meta * 0.999]['iter'].iloc[0] + 1

        # Compute acceleration for each method
        metodos = ['PGD', 'FISTA', 'Momentum', 'AA_m3', 'AA_m4', 'AA_m5', 'RL Agent']
        resultados = []

        for nombre_met in metodos:
            df_met = df[(df['image'] == idx_imagen) & (df['method'] == nombre_met)].sort_values('iter')
            logro = df_met[df_met[m] >= valor_meta * 0.999]

            if not logro.empty:
                iter_logro = logro['iter'].iloc[0] + 1
            else:
                iter_logro = 1000
            # Acceleration Factor (AF)
            af = t_ref / iter_logro

            resultados.append({
                'Método': nombre_met,
                'Valor Meta': round(valor_meta, 4),
                'Iteración': iter_logro,
                'AF': f"{af:.2f}x"
            })

        print(f"\n>>> Scenario: {esc['label']} (Metric: {m.upper()})")
        df_res = pd.DataFrame(resultados).set_index('Método')
        print(df_res)
        print("-" * 60)

    except FileNotFoundError:
        print(f"\n[!] Error: No se encontró el archivo {esc['archivo']}")
    except Exception as e:
        print(f"\n[!] Error procesando {esc['label']}: {e}")

Reports image index: 3

>>> Scenario: PGD CR 0.3 (Metric: PSNR)
          Valor Meta  Iteración      AF
Método                                 
PGD          31.8528        197   1.00x
FISTA        31.8528        108   1.82x
Momentum     31.8528        105   1.88x
AA_m3        31.8528         38   5.18x
AA_m4        31.8528         40   4.92x
AA_m5        31.8528         41   4.80x
RL Agent     31.8528         15  13.13x
------------------------------------------------------------

>>> Scenario: PGD CR 0.1 (Metric: PSNR)
          Valor Meta  Iteración     AF
Método                                
PGD          25.3902        212  1.00x
FISTA        25.3902        116  1.83x
Momentum     25.3902        114  1.86x
AA_m3        25.3902         41  5.17x
AA_m4        25.3902         44  4.82x
AA_m5        25.3902         44  4.82x
RL Agent     25.3902         31  6.84x
------------------------------------------------------------

>>> Scenario: ADMM CR 0.3 (Metric: SSIM)
          Valor Meta

In [ ]:
# Files
archivos = {
    'PGD_03': escenarios[0]['archivo'],
    'PGD_01': escenarios[1]['archivo'],
    'ADMM_03': escenarios[2]['archivo'],
    'ADMM_01': escenarios[3]['archivo']
}

def generar_tabla_rendimiento(file03, file01, titulo):

    df03 = pd.read_csv(file03)
    df01 = pd.read_csv(file01)

    def promediar_set5(df):

        ultima_iter = df[df['iter'] == 999]

        res = ultima_iter.groupby('method').agg({
            'psnr': 'mean',
            'ssim': 'mean',
            'error_gt': 'mean'
        }).reindex(['PGD', 'FISTA', 'Momentum', 'AA_m3', 'AA_m4', 'AA_m5', 'RL Agent'])
        return res

    res03 = promediar_set5(df03)
    res01 = promediar_set5(df01)

    tabla_final = pd.concat([res03, res01], axis=1, keys=['CR 0.3', 'CR 0.1'])

    print(f"\n>>> General Performance: {titulo}")
    print(tabla_final.round(4))
    print("-" * 80)

# --- EJECUCIÓN ---
print("Tables of performance (Set5)")
print("="*80)

# Tabla para PGD
generar_tabla_rendimiento(archivos['PGD_03'], archivos['PGD_01'], "PnP-PGD")

# Tabla para ADMM
generar_tabla_rendimiento(archivos['ADMM_03'], archivos['ADMM_01'], "PnP-ADMM")

Tables of performance (Set5)

>>> General Performance: PnP-PGD
           CR 0.3                    CR 0.1                 
             psnr    ssim error_gt     psnr    ssim error_gt
method                                                      
PGD       32.3611  0.9348   0.0607  26.8141  0.8052   0.1214
FISTA     32.3257  0.9324   0.0609  26.8119  0.8032   0.1212
Momentum  32.3545  0.9343   0.0607  26.8263  0.8054   0.1211
AA_m3     32.3606  0.9348   0.0607  26.8172  0.8055   0.1213
AA_m4     32.3603  0.9348   0.0607  26.8176  0.8055   0.1213
AA_m5     32.3599  0.9348   0.0607  26.8170  0.8055   0.1213
RL Agent  33.1602  0.9469   0.0558  27.0597  0.8155   0.1186
--------------------------------------------------------------------------------

>>> General Performance: PnP-ADMM
           CR 0.3                    CR 0.1                 
             psnr    ssim error_gt     psnr    ssim error_gt
method                                                      
PGD       32.9445  0.9393   